# Classification Track (Part A) — Sentiment of Restaurant Reviews

**23CSE301 Machine Learning Capstone · Review 1**
**Project:** *Predictive Analytics Across Domains: Student Performance, Sentiment Classification and Customer Segmentation*

| | |
|---|---|
| **Problem** | Predict whether a restaurant review is **positive** or **negative** from its text alone. |
| **Task type** | Supervised learning, **binary text classification**. |
| **Dataset** | *10000 Restaurant Reviews*, Kaggle — 10,000 rows × 8 columns. |
| **Track owner** | _(student name)_ |

### Review 1 scope — Part A only
This notebook implements the **five Part A classifiers** of guideline 3.2: Logistic Regression,
K-Nearest Neighbors, Gaussian Naive Bayes, Decision Tree and Support Vector Machine.
**Part B is not part of Review 1 and is not implemented here.** When Part B is added in Review 2 it
will reuse the same label definition, split and seed, so the Part A numbers will not change.

### Metric conventions (fixed for the whole notebook)
* The label is **derived** from the star rating: rating ≥ 4 → `positive`, rating ≤ 2 → `negative`,
  ratings strictly between 2 and 4 are dropped as ambiguous.
* **`pos_label = "negative"`.** `negative` is the minority class, so binary precision, recall and F1
  are reported **for the `negative` class** and labelled that way in every table.
* **Weighted** precision / recall / F1 average the two classes weighted by their support.
* **ROC-AUC** is computed from continuous scores (`predict_proba` or `decision_function`), never from
  hard predicted labels. The table records which score each model used.
* A **majority-class baseline** is shown beside the results for reference.

Cells marked **✍️ TEAM ANALYSIS REQUIRED** are for the team's own observations and must not be
written by an AI tool.

## 1. Set-up

**What:** import libraries, locate the project root, apply the plotting style and print the environment.

**Why:** reproducibility — the printed versions and seed identify exactly how these results were made.

In [1]:
# --- Environment set-up --------------------------------------------------------
import sys
import platform
from pathlib import Path

# Find the project root (the folder containing src/config.py). This works whether
# the notebook is opened from notebooks/ or from the project root, and avoids any
# hard-coded absolute path.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from src import config, show_source
from src.plotting import set_style

set_style()                                   # titles, labels, colourblind palette
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Only folder NAMES are printed, so no personal absolute paths end up in the committed notebook.
print(f"Python        {platform.python_version()}  (environment: {Path(sys.prefix).name})")
print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} "
      f"| matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")
print(f"Project root  .../{ROOT.name}")
print(f"Random seed   {config.RANDOM_STATE}   test size {config.TEST_SIZE}   CV folds {config.CV_FOLDS}")
print(f"Run mode      {config.RUN_MODE}" + ("   (DEV RUN - outputs carry the '_dev' suffix, NOT reported results)"
                                             if config.RUN_MODE == "dev" else ""))

Python        3.13.2  (environment: ml-capstone)
numpy 2.5.3 | pandas 3.0.6 | scikit-learn 1.9.1 | matplotlib 3.11.2 | seaborn 0.13.2
Project root  .../Machine Learning project
Random seed   42   test size 0.2   CV folds 5
Run mode      full


In [2]:
# --- Is the raw data present and unchanged? ------------------------------------
# load_raw() verifies the SHA-256 checksum, row count and exact column list against
# the dataset contract in src/config.py and raises ContractError on any mismatch.
from src.data_loading import load_raw

contract = config.REVIEWS
_df_check = load_raw(contract)
print(f"{contract.filename}: {_df_check.shape[0]:,} rows x {_df_check.shape[1]} columns - checksum and schema match the contract")
del _df_check

Restaurant reviews.csv: 10,000 rows x 8 columns - checksum and schema match the contract


## 2. Data loading and audit  *(rubric A1)*

**What:** load the raw CSV and report shape, column types, missing values, unique counts and duplicates.

**Why:** the raw file has features that need attention before modelling — the rating is stored as
text, one column is a scraping artefact, and some rows have no review text. The audit makes each of
these visible so that every later decision is based on the real data.

**Concept — leakage columns and identifiers:** a column that records the answer (here, `Rating`) or
identifies the source (`Restaurant`, `Reviewer`) must not be used as a model input. The excluded columns
and the reason for each are listed in `src/config.py` and `docs/dataset_sources.md`.

> 🔧 **Build status:** the code for this section is added in **Phase 6**. This placeholder is replaced when that phase is built.

## 3. Deriving the sentiment label from the rating

**What:** convert each star rating into `positive` (≥ 4) or `negative` (≤ 2) and drop the rest.

**Why:** the dataset has no sentiment column — only a 1–5 rating (with some half-stars, one
non-numeric value `"Like"`, and missing values). Middle ratings (2.5, 3, 3.5) express mixed opinions;
dropping them gives a cleaner binary problem. The table in this section reports how every raw rating
value was handled, with counts, and the resulting class balance.

**Concept — label derivation:** when a task's label does not exist directly, it is defined by a rule
applied to another column. The rule is decided before modelling and kept fixed, otherwise results
across reviews would not be comparable.

> 🔧 **Build status:** the code for this section is added in **Phase 6**. This placeholder is replaced when that phase is built.

## 4. Cleaning: missing text, artefact column and duplicates  *(rubric B1)*

**What:** drop the artefact column, drop rows without review text, and investigate duplicate rows
under several candidate keys before deciding which to remove.

**Why:** a row without text cannot be classified by a text model. Exact duplicate rows would be
counted twice and could appear in both the training and the test set. Short repeated texts such as
"good", however, are written by different reviewers and are legitimate separate observations.

These steps learn nothing from the data (no statistics are computed), so they can safely run before
the train/test split.

> 🔧 **Build status:** the code for this section is added in **Phase 6**. This placeholder is replaced when that phase is built.

> ### ✍️ TEAM ANALYSIS REQUIRED — `CLF-B1`: Cleaning decisions and their justification
> **Author:** _(Classification track owner — write your name)_
>
> This cell must be written by the team in your own words. Course guideline 7.5 does not allow
> generative-AI tools to write analysis or interpretation.
>
> **Guiding questions** (base every answer on the outputs above):
> - Which duplicate key did you choose and why? How many rows did it remove?
> - Why were repeated short texts (e.g. "good") kept? Do repeated texts with different ratings worry you?
> - What did you decide about rows with no review text, and why?
>
> *When you have written your answer, replace this whole cell with your text under the heading
> `### Team analysis — CLF-B1`. `scripts/validate_project.py` counts the cells that still contain the marker.*

## 5. Stratified train/test split  *(rubric B2)*

**What:** split the labelled reviews once into 80 % training and 20 % test, **stratified** by label.

**Why:** stratification keeps the positive/negative proportion identical in both sets, so the test
set is representative of the minority class. The split happens before EDA and before the TF-IDF
vocabulary is learned, so neither can be influenced by the test set.

> 🔧 **Build status:** the code for this section is added in **Phase 6**. This placeholder is replaced when that phase is built.

## 6. Exploratory data analysis on the training set  *(rubric A2, A3)*

**What:** class balance, review length by class, distributions of the numeric metadata (pictures,
reviewer activity), a correlation heatmap, feature–target scatter/box plots, and the most frequent
words and word pairs per class.

**Why:** the model will only see the text, but these views show how the classes differ and what
signal is available. The rubric's "distribution plot for each feature" is interpreted for text data
as the distributions of the numeric attributes and of text length (see `docs/clarifications.md`).

Each plot is followed by a ✍️ team-observation cell.

> 🔧 **Build status:** the code for this section is added in **Phase 6**. This placeholder is replaced when that phase is built.

## 7. Text representation: TF-IDF  *(rubric B2)*

**What:** convert each review into a numeric vector with TF-IDF, learned from the training reviews only.

**Why:** classifiers need numbers. TF-IDF gives each word or word pair a weight that is high when it is
frequent in this review but rare across reviews. All five classifiers receive the **same** TF-IDF
representation so the comparison is fair.

**Concept — TF-IDF:** *term frequency × inverse document frequency*. Each review vector is then
L2-normalised to unit length, which acts as the scaling step for distance- and margin-based models.
No stock stop-word list is used, because it would delete negations such as "not" and "no".

**Size of the representation:** before fixing the vocabulary cap (`max_features`), this section
measures the real vocabulary size, the memory a dense copy would need (Gaussian Naive Bayes requires
dense input) and the runtime.

> 🔧 **Build status:** the code for this section is added in **Phase 6**. This placeholder is replaced when that phase is built.

## 8. Feature engineering  *(rubric B3)*

**What:** add at least one engineered feature and measure its effect.

**How it works here:** the team registers the feature in `src/feature_engineering.py`
(*TEAM REGISTRATIONS*). It is computed row by row inside the pipeline, so it cannot leak test
information. This section shows results with and without it.

> 🔧 **Build status:** the code for this section is added in **Phase 6**. This placeholder is replaced when that phase is built.

> ### ✍️ TEAM ANALYSIS REQUIRED — `CLF-B3`: Engineered feature and justification
> **Author:** _(Classification track owner — write your name)_
>
> This cell must be written by the team in your own words. Course guideline 7.5 does not allow
> generative-AI tools to write analysis or interpretation.
>
> **Guiding questions** (base every answer on the outputs above):
> - Which feature did you create, and from which column(s)?
> - Why might it help distinguish positive from negative reviews?
> - Is it computed row by row, so that it cannot leak test information?
> - After measuring: did it help? Report the numbers, including if it did not.
>
> *When you have written your answer, replace this whole cell with your text under the heading
> `### Team analysis — CLF-B3`. `scripts/validate_project.py` counts the cells that still contain the marker.*

## 9. Majority-class baseline

**What:** the scores of a "model" that always predicts the most frequent training class.

**Why:** with imbalanced classes, accuracy alone is misleading — always answering `positive` already
scores around the positive-class share. Every real model must be judged against this reference. It is
a reference point, **not** one of the five Part A algorithms.

> 🔧 **Build status:** the code for this section is added in **Phase 7**. This placeholder is replaced when that phase is built.

## 10. The five Part A classifiers  *(rubric D1, D2)*

**What:** train each Part A classifier on the same TF-IDF training data and evaluate it on the same
test set, reporting accuracy, precision, recall, weighted F1, ROC-AUC and the confusion matrix.

**How each subsection is organised:** what the algorithm is; how it works step by step; the settings
worth tuning; a likely viva question; the code that builds, trains and records it; and its specific
output.

### 10.1 Logistic Regression

*In one line:* models the log-odds of the `negative` class as a weighted sum of word weights; coefficients convert to odds ratios.

> 🔧 **Build status:** implemented in **Phase 7**.

### 10.2 K-Nearest Neighbors

*In one line:* labels a review by majority vote of the *k* most similar training reviews; tuned through `k` and the distance metric.

> 🔧 **Build status:** implemented in **Phase 7**.

### 10.3 Gaussian Naive Bayes

*In one line:* applies Bayes' theorem assuming features are conditionally independent and normally distributed within each class; requires dense input.

> 🔧 **Build status:** implemented in **Phase 7**.

### 10.4 Decision Tree Classifier

*In one line:* splits on word weights to separate the classes; tuned through `max_depth`; the tree is visualised.

> 🔧 **Build status:** implemented in **Phase 7**.

### 10.5 Support Vector Machine (SVC)

*In one line:* finds the maximum-margin boundary between the classes; tuned through `C` and `kernel`; ROC-AUC uses `decision_function`.

> 🔧 **Build status:** implemented in **Phase 7**.

## 11. Part A comparison table  *(rubric D2)*

**What:** one table with all six metrics for the five classifiers, ranked by weighted F1, with the
majority-class baseline for reference.

**Why weighted F1 for ranking:** it balances precision and recall across both classes, weighted by
their size, and is the metric the rubric names. Minority-class (`negative`) recall is shown separately
so that a model which ignores the minority class cannot hide behind high accuracy.

> 🔧 **Build status:** the code for this section is added in **Phase 8**. This placeholder is replaced when that phase is built.

## 12. Stratified 5-fold cross-validation  *(guideline 7.2)*

**What:** 5-fold cross-validated weighted F1 on the **training set** for all five classifiers, used to
nominate the two leaders.

**Why:** cross-validation estimates performance more reliably than a single split, and doing it on the
training data keeps the test set untouched. The TF-IDF vocabulary is re-learned inside every fold.

> 🔧 **Build status:** the code for this section is added in **Phase 8**. This placeholder is replaced when that phase is built.

## 13. Hyperparameter tuning of the two leaders

**What:** `GridSearchCV` on the two leading classifiers with before/after test metrics.

**Why:** model selection uses cross-validation on the training set only; the test set is scored once
afterwards. If tuning improves one metric and harms another, both directions are reported.

> 🔧 **Build status:** the code for this section is added in **Phase 8**. This placeholder is replaced when that phase is built.

## 14. Confusion matrices and ROC curves

**What:** confusion matrices for all five classifiers side by side, and their ROC curves on one plot.

**Why:** the confusion matrix shows *which* mistakes a model makes (missed negative reviews versus
false alarms); the ROC curve shows the trade-off between them across all decision thresholds.

> 🔧 **Build status:** the code for this section is added in **Phase 8**. This placeholder is replaced when that phase is built.

## 15. Saving the fitted pipelines

**What:** save each fitted pipeline (TF-IDF + classifier) to `models/` with compressed `joblib`.

> 🔧 **Build status:** the code for this section is added in **Phase 8**. This placeholder is replaced when that phase is built.

## 16. Model selection and conclusions

> ### ✍️ TEAM ANALYSIS REQUIRED — `CLF-CONCLUSION`: Model selection, conclusions and limitations
> **Author:** _(Classification track owner — write your name)_
>
> This cell must be written by the team in your own words. Course guideline 7.5 does not allow
> generative-AI tools to write analysis or interpretation.
>
> **Guiding questions** (base every answer on the outputs above):
> - Which Part A classifier performs best on this problem, and by which metrics? How does it compare with the majority-class baseline?
> - Why do you think Gaussian Naive Bayes and KNN behave as they do on TF-IDF features?
> - Which errors (missed negative reviews vs. false alarms) matter more for this problem, and which model handles them best?
> - What did tuning change? Note that Review 2 (Part B) may change which model leads.
> - What are the limitations — label derivation from ratings, dropped middle ratings, duplicate texts?
>
> *When you have written your answer, replace this whole cell with your text under the heading
> `### Team analysis — CLF-CONCLUSION`. `scripts/validate_project.py` counts the cells that still contain the marker.*

## 17. References and AI-assistance disclosure

* 23CSE301 Machine Learning — Capstone Project Guidelines, Algorithm List & Rubrics (2026-27).
* Dataset: J. Arvidsson, *10000 Restaurant Reviews*, Kaggle —
  https://www.kaggle.com/datasets/joebeachcapital/restaurant-reviews
  (upstream: https://github.com/manthanpatel98/Restaurant-Review-Sentiment-Analysis; see
  `docs/dataset_sources.md` for checksum and licence notes).
* scikit-learn documentation — https://scikit-learn.org/stable/

**AI assistance (guideline 7.5):** an AI coding assistant generated the code scaffolding, the
supporting `src/` modules and the factual algorithm explanations. It did **not** write the EDA
observations, the feature-engineering justification, the model-selection reasoning or the conclusions
— those are the ✍️ team cells. Full disclosure: `README.md`.